In [1]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

import os
os.environ['TRANSFORMERS_CACHE'] = './cache/'
import transformers
from torch import nn
import torch
from be_great.multihead_models import MOEModelForCausalLM
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, Trainer, TrainingArguments, EarlyStoppingCallback, BitsAndBytesConfig
from torch.optim import AdamW
from torch.optim.lr_scheduler import LinearLR
from matplotlib import pyplot as plt
from tqdm import tqdm 
import argparse
import datetime
import json
from be_great import GReaT
from be_great.great_dataset import GReaTDataset, GReaTDataCollator
from be_great.great_trainer import GReaTTrainer
import re
from shutil import copy
from sklearn import preprocessing, pipeline, ensemble, compose

/home/sonia/miniconda3/envs/greatt/lib/python3.11/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [2]:
modelname = 'meta-llama/Meta-Llama-3-8B'
tokenizer = AutoTokenizer.from_pretrained(modelname, padding_side='left')
special_tokens_dict = {"bos_token": "<BOS>", 'eos_token': '<EOS>'}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

In [3]:
tokenizer.eos_token_id

128257

In [4]:
dgpt2 = transformers.AutoModelForCausalLM.from_pretrained(modelname, device_map='auto',
            quantization_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", 
            bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16))

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [5]:
great= GReaT.load_from_dir('/home/sonia/tabby/ckpts/llamamh/great/1e-6/1',
                           model = dgpt2)

In [6]:
great.model.resize_token_embeddings(len(tokenizer))

Embedding(128258, 4096)

In [7]:
great.model = MOEModelForCausalLM(great.model, num_experts=6, moe=False, multihead=True)

In [8]:
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    TaskType,
)

def apply_efficient_finetuning(model):
    lora_config = LoraConfig(
        r=1,  
        lora_alpha=256,
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'down_proj', 'up_proj', 'lm_head.layers.0', 'lm_head.layers.1','lm_head.layers.2', 'lm_head.layers.3', 'lm_head.layers.4', 'lm_head.layers.5'],
        lora_dropout=0.05,
        bias="none",
        task_type=TaskType.CAUSAL_LM,  # this is specific for gpt2 model, to be adapted
    )
    # prepare int-8 model for training
    model = prepare_model_for_kbit_training(model)
    # add LoRA adaptor
    model = get_peft_model(model, lora_config)
    model.print_trainable_parameters()
    return model
    print('applying lora, model type now', type(model))

In [9]:
great.model = apply_efficient_finetuning(great.model)

trainable params: 3,415,564 || all params: 10,660,417,036 || trainable%: 0.0320


In [10]:
ckpt_path = '/home/sonia/tabby/ckpts/llamamh/great/1e-6/1/model.pt'
sd = torch.load(ckpt_path)
# dgpt2copy.load_state_dict(torch.load(ckpt_path, weights_only=True))
sd.keys()

/tmp/ipykernel_132313/2087372576.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  sd = torch.load(ckpt_path)


odict_keys(['base_model.model.model.embed_tokens.weight', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight.absmax', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight.quant_map', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight.nested_absmax', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight.nested_quant_map', 'base_model.model.model.layers.0.self_attn.q_proj.base_layer.weight.quant_state.bitsandbytes__nf4', 'base_model.model.model.layers.0.self_attn.q_proj.lora_A.default.weight', 'base_model.model.model.layers.0.self_attn.q_proj.lora_B.default.weight', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight.absmax', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight.quant_map', 'base_model.model.model.layers.0.self_attn.k_proj.base_layer.weight.n

In [11]:
for name, param in great.model.named_parameters():
    param.data.copy_(sd[name])

In [12]:
great.model.eval() 
great.tokenizer = tokenizer
# columns = ['bedrooms','occupancy','value_median_house']
# column_names_tokens = tokenizer(columns).input_ids
# great.model.set_generation_mode(token_heads=list(range(6)), column_names_tokens=column_names_tokens)

In [13]:
great.sample(10, k=1, max_length = 1000)

  0%|          | 0/10 [00:00<?, ?it/s]

100%|██████████| 10/10 [00:13<00:00,  1.38s/it]


['<|begin_of_text|>value_median_house is 2.58100<EOS><|begin_of_text|>rooms is 5.5<EOS><|begin_of_text|>income_median is 3.9062<EOS><|begin_of_text|>age_median is 34.0<EOS><|begin_of_text|>occupancy is 2.15<EOS><|begin_of_text|>bedrooms is 1.0<EOS>',
 '<|begin_of_text|>value_median_house is 0.88200<EOS><|begin_of_text|>rooms is 6.2<EOS><|begin_of_text|>income_median is 2.5<EOS><|begin_of_text|>bedrooms is 1.1<EOS><|begin_of_text|>occupancy is 2.9<EOS><|begin_of_text|>age_median is 24.0<EOS>',
 '<|begin_of_text|>value_median_house is 1.55700<EOS><|begin_of_text|>occupancy is 3.125<EOS><|begin_of_text|>bedrooms is 1.0625<EOS><|begin_of_text|>rooms is 4.6875<EOS><|begin_of_text|>age_median is 42.0<EOS><|begin_of_text|>income_median is 2.2159<EOS>',
 '<|begin_of_text|>value_median_house is 1.64900<EOS><|begin_of_text|>bedrooms is 1.008<EOS><|begin_of_text|>occupancy is 4.36<EOS><|begin_of_text|>income_median is 3.5833<EOS><|begin_of_text|>rooms is 5.04<EOS><|begin_of_text|>age_median is 35